In [3]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, make_scorer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import pickle

Load trained models and rest data

In [51]:
target_train = pd.read_parquet("../data/target_train.parquet")
network_train = pd.read_parquet("../data/network_train.parquet")
weather_train = pd.read_parquet("../data/weather_train.parquet")
weather_test = pd.read_parquet("../data/weather_test.parquet")
network_test = pd.read_parquet("../data/network_test.parquet")

In [4]:
with open('../models/xgboost_solar_model.pkl', 'rb') as f:
    solar_model = pickle.load(f)
with open('../models/tuned_lgbm_wind_model.pkl', 'rb') as f:
    wind_model = pickle.load(f)
with open('../models/load_final.pkl', 'rb') as f:
    load_model = pickle.load(f)

# Load features saved from notebooks 02, 03, 04
X_train_solar = pd.read_parquet('../data/X_train_solar.parquet')
X_valid_solar = pd.read_parquet('../data/X_valid_solar.parquet')
y_train_solar = pd.read_parquet('../data/y_train_solar.parquet').squeeze()

X_train_wind = pd.read_parquet('../data/X_train_wind.parquet')
X_valid_wind = pd.read_parquet('../data/X_valid_wind.parquet')
y_train_wind = pd.read_parquet('../data/y_train_wind.parquet').squeeze()

X_train_load = pd.read_parquet('../data/X_train_load.parquet')
X_valid_load = pd.read_parquet('../data/X_valid_load.parquet')
y_train_load = pd.read_parquet('../data/y_train_load.parquet').squeeze()

print(f"Solar  — train: {X_train_solar.shape}, valid: {X_valid_solar.shape}")
print(f"Wind   — train: {X_train_wind.shape}, valid: {X_valid_wind.shape}")
print(f"Load   — train: {X_train_load.shape}, valid: {X_valid_load.shape}")

Solar  — train: (35060, 28), valid: (8765, 28)
Wind   — train: (34640, 22), valid: (8661, 22)
Load   — train: (35011, 20), valid: (8749, 20)


In [7]:
## Find common indices
common_idx = (X_train_solar.index
              .intersection(X_train_wind.index)
              .intersection(X_train_load.index))

print(f"\nCommon indices: {len(common_idx)} rows")

# Keep only common indices in all three
X_train_solar = X_train_solar.loc[common_idx]
y_train_solar = y_train_solar.loc[common_idx]

X_train_wind = X_train_wind.loc[common_idx]
y_train_wind = y_train_wind.loc[common_idx]

X_train_load = X_train_load.loc[common_idx]
y_train_load = y_train_load.loc[common_idx]

print("\nAfter alignment:")
print(f"  X_train_solar: {X_train_solar.shape}")
print(f"  X_train_wind: {X_train_wind.shape}")
print(f"  X_train_load: {X_train_load.shape}")



Common indices: 34610 rows

After alignment:
  X_train_solar: (34610, 28)
  X_train_wind: (34610, 22)
  X_train_load: (34610, 20)


Stacking /  Using models' predictions as features for price model

In [13]:
# Generate OOF predictions-So re-generate it with cross-validation:

def generate_oof_with_your_model_architecture():
    """
    I use your the same model architecture (XGBoost, LightGBM, etc) used in  notebooks 02 03 04
   
    """
    
    tscv = TimeSeriesSplit(n_splits=3)
    oof_solar = np.zeros(len(X_train_solar))
    oof_wind = np.zeros(len(X_train_wind))
    oof_load = np.zeros(len(X_train_load))
    
    for fold, (train_idx, valid_idx) in enumerate(tscv.split(X_train_solar)):
        print(f"Fold {fold + 1}:")
        
        #  SOLAR
        X_tr_solar = X_train_solar.iloc[train_idx]
        y_tr_solar = y_train_solar.iloc[train_idx]
        X_val_solar = X_train_solar.iloc[valid_idx]
        
      
        fold_solar =solar_model
        fold_solar.fit(X_tr_solar, y_tr_solar)
        oof_solar[valid_idx] = fold_solar.predict(X_val_solar)
        
        # WIND 
        X_tr_wind = X_train_wind.iloc[train_idx]
        y_tr_wind = y_train_wind.iloc[train_idx]
        X_val_wind = X_train_wind.iloc[valid_idx]
        
        fold_wind = wind_model
        fold_wind.fit(X_tr_wind, y_tr_wind)
        oof_wind[valid_idx] = fold_wind.predict(X_val_wind)
        
        # LOAD 
        X_tr_load = X_train_load.iloc[train_idx]
        y_tr_load = y_train_load.iloc[train_idx]
        X_val_load = X_train_load.iloc[valid_idx]
        
        fold_load = load_model
        fold_load.fit(X_tr_load, y_tr_load)
        oof_load[valid_idx] = fold_load.predict(X_val_load)
        
        print(f"  Solar fold: {len(valid_idx)} samples")
        print(f"  Wind fold: {len(valid_idx)} samples")
        print(f"  Load fold: {len(valid_idx)} samples")
    
    return oof_solar, oof_wind, oof_load

# Generate OOF
oof_solar, oof_wind, oof_load = generate_oof_with_your_model_architecture()
print(f" OOF generated: {len(oof_solar)} predictions each")

Fold 1:
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4650
[LightGBM] [Info] Number of data points in the train set: 8654, number of used features: 22
[LightGBM] [Info] Start training from score 4373.038710
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

In [17]:




def flatten_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = ["_".join([str(i) for i in col]) for col in df.columns]
    return df


def interpolate_weather(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_index()
    df = df.interpolate(method="time")
    df = df.ffill()
    return df


def aggregate_weather(df: pd.DataFrame) -> pd.DataFrame:
    ssrd_cols = [c for c in df.columns if c.endswith("ssrd")]
    tcc_cols  = [c for c in df.columns if c.endswith("tcc")]
    temp_cols = [c for c in df.columns if c.endswith("2t")]
    wind_cols = [c for c in df.columns if c.endswith("100ws")]

    flat = pd.DataFrame(index=df.index)
    flat["ssrd_mean"] = df[ssrd_cols].mean(axis=1)
    flat["ssrd_std"]  = df[ssrd_cols].std(axis=1)
    flat["tcc_mean"]  = df[tcc_cols].mean(axis=1)
    flat["tcc_std"]   = df[tcc_cols].std(axis=1)
    flat["temp_mean"] = df[temp_cols].mean(axis=1)
    flat["temp_std"] = df[temp_cols].std(axis=1)
    flat["wind_mean"] = df[wind_cols].mean(axis=1)
    flat["wind_std"]  = df[wind_cols].std(axis=1)

    return flat.bfill().ffill()


def prepare_weather(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten multi-index columns, interpolate missing values, and aggregate to per-hour stats."""
    df = flatten_weather_columns(df)
    df = interpolate_weather(df)
    df = aggregate_weather(df)
    df = df.reset_index().rename(columns={df.index.name or "index": "ds"})
    return df


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ssrd_lag1"]  = df["ssrd_mean"].shift(1)
    df["ssrd_lag24"] = df["ssrd_mean"].shift(24)
    return df


def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ssrd_roll_3h"]     = df["ssrd_mean"].rolling(3).mean()
    df["ssrd_roll_24h"]    = df["ssrd_mean"].rolling(24).mean()
    df["tcc_roll_6h_std"]  = df["tcc_mean"].rolling(6).std()

    # Add new rolling features for tcc_mean, temp_mean, and wind_mean
    df["tcc_roll_3h_mean"] = df["tcc_mean"].rolling(3).mean()
    df["tcc_roll_24h_mean"] = df["tcc_mean"].rolling(24).mean()
    df["temp_roll_3h_mean"] = df["temp_mean"].rolling(3).mean()
    df["temp_roll_24h_mean"] = df["temp_mean"].rolling(24).mean()
    df["wind_roll_3h_mean"] = df["wind_mean"].rolling(3).mean()
    df["wind_roll_24h_mean"] = df["wind_mean"].rolling(24).mean()
    df["temp_roll_6h_std"]  = df["temp_mean"].rolling(6).std()
    df["wind_roll_6h_std"]  = df["wind_mean"].rolling(6).std()

    return df


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(df["ds"])
    df["hour"]      = dt.dt.hour
    df["month"]     = dt.dt.month
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_solar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ssrd_norm"]       = df["ssrd_mean"] / df["ssrd_mean"].max()
    df["solar_potential"] = df["ssrd_norm"] * (1 - df["tcc_mean"])
    return df


def engineer_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lags, rolling stats, time encodings, and solar features, then drop NaN rows."""
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_time_features(df)



    df = add_solar_features(df)
    return df.dropna()
def prepare_solar_data(target_path: str, weather_path: str, network_path: str) -> pd.DataFrame:
    """Complete pipeline: load + preprocess weather + merge with targets and network features."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)
    network_train = pd.read_parquet(network_path)

    # Interpolate and clip FR_solar_actual before merging
    target_train["FR_solar_actual"] = (
        target_train["FR_solar_actual"]
        .interpolate(method="time")
        .bfill()
        .ffill()
        .clip(lower=0)
    )

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    # Merge with targets and network features
    solar_df = target_train[['FR_solar_actual']].join(weather_features, how='inner').join(network_train, how='inner')

    return solar_df

def prepare_wind_data(target_path: str, weather_path: str) -> pd.DataFrame:
    """Complete pipeline for wind."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    wind_df = target_train[['FR_wind_actual']].join(weather_features)

    return wind_df

def prepare_price_data(target_path: str, weather_path: str, network_path: str) -> pd.DataFrame:
    """Complete pipeline for load (includes network features)."""
    target_train = pd.read_parquet(target_path)
    weather_train = pd.read_parquet(weather_path)
    network_train = pd.read_parquet(network_path)

    weather_features = engineer_weather_features(prepare_weather(weather_train))

    price_df = target_train[['FR_price_actual']].join(weather_features).join(network_train)

    return price_df


In [18]:
price_df=prepare_price_data("../data/target_train.parquet","../data/weather_train.parquet","../data/network_train.parquet")

In [19]:
price_df.columns

Index(['FR_price_actual', 'ds', 'ssrd_mean', 'ssrd_std', 'tcc_mean', 'tcc_std',
       'temp_mean', 'temp_std', 'wind_mean', 'wind_std', 'ssrd_lag1',
       'ssrd_lag24', 'ssrd_roll_3h', 'ssrd_roll_24h', 'tcc_roll_6h_std',
       'tcc_roll_3h_mean', 'tcc_roll_24h_mean', 'temp_roll_3h_mean',
       'temp_roll_24h_mean', 'wind_roll_3h_mean', 'wind_roll_24h_mean',
       'temp_roll_6h_std', 'wind_roll_6h_std', 'hour', 'month', 'hour_sin',
       'hour_cos', 'month_sin', 'month_cos', 'ssrd_norm', 'solar_potential',
       'EEX_CARBON', 'EEX_COAL', 'EEX_GAS_PEG', 'FR_capacity_solar',
       'FR_capacity_wind', 'FR_availability_coal', 'FR_availability_gas',
       'FR_availability_hydro', 'FR_availability_nuclear'],
      dtype='object')

In [20]:
y_train_price=price_df['FR_price_actual']

In [22]:

X_price_train = price_df[[ 
  
    'temp_mean', 'temp_std', 
    'temp_roll_3h_mean', 'temp_roll_24h_mean',
    'tcc_mean', 'tcc_std', 
    'tcc_roll_3h_mean', 'tcc_roll_24h_mean', 'tcc_roll_6h_std',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'EEX_CARBON', 'EEX_COAL', 'EEX_GAS_PEG',
    'FR_availability_coal', 'FR_availability_gas', 
    'FR_availability_hydro', 'FR_availability_nuclear',
    
]]


In [26]:
oof_index = X_train_solar.index

print(f"OOF index length: {len(oof_index)}")
print(f"Price DF original length: {len(price_df)}")


OOF index length: 34610
Price DF original length: 43849


I need to use the intersection of both

In [27]:
## Create OOF DataFrame with proper index
oof_df = pd.DataFrame({
    'pred_solar': pd.Series(oof_solar, index=oof_index),
    'pred_wind':  pd.Series(oof_wind,  index=oof_index),
    'pred_load':  pd.Series(oof_load,  index=oof_index),
})

# Inner join keeps only matching indices
X_price_train = X_price_train.join(oof_df, how='inner')

print(f"X_price_train after join: {len(X_price_train)} rows")
print(f"Columns ({len(X_price_train.columns)}): {X_price_train.columns.tolist()}")

# Get target (aligned to X_price_train's index)
y_price_train = price_df.loc[X_price_train.index, 'FR_price_actual']


print(f"  X_price_train: {X_price_train.shape}")
print(f"  y_price_train: {y_price_train.shape}")

# Check for NaN
nan_count = X_price_train.isnull().sum().sum()


X_price_train after join: 34610 rows
Columns (23): ['temp_mean', 'temp_std', 'temp_roll_3h_mean', 'temp_roll_24h_mean', 'tcc_mean', 'tcc_std', 'tcc_roll_3h_mean', 'tcc_roll_24h_mean', 'tcc_roll_6h_std', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'EEX_CARBON', 'EEX_COAL', 'EEX_GAS_PEG', 'FR_availability_coal', 'FR_availability_gas', 'FR_availability_hydro', 'FR_availability_nuclear', 'pred_solar', 'pred_wind', 'pred_load']
  X_price_train: (34610, 23)
  y_price_train: (34610,)


In [28]:
price_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=50,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    verbose=-1
)

price_model.fit(X_price_train, y_price_train)


,boosting_type,'gbdt'
,num_leaves,50
,max_depth,7
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [29]:
# Check feature importance
importance_df = pd.DataFrame({
    'feature': X_price_train.columns,
    'importance': price_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 features:")
print(importance_df.head(10))



Top 10 features:
                    feature  importance
22                pred_load        2007
19  FR_availability_nuclear        1836
15              EEX_GAS_PEG        1526
17      FR_availability_gas        1523
13               EEX_CARBON        1517
20               pred_solar        1452
14                 EEX_COAL        1431
18    FR_availability_hydro        1398
21                pred_wind        1289
16     FR_availability_coal         305


For price prediction models, feature importance reveals which factors the model considers most critical when forecasting electricity prices—in this case, demand (pred_load) emerging as the strongest driver with an importance score of 2,007, followed by nuclear availability (1,836) and gas prices (1,526). This ranking aligns well with economic reality: electricity prices are primarily determined by the balance between supply and demand, with natural gas serving as the marginal cost setter in most markets. The presence of prediction features (pred_load, pred_solar, pred_wind) in the top rankings—collectively accounting for 33% of the model's decision power—validates the stacking approach, demonstrating that using trained model predictions as input features provides meaningful signal compared to raw weather data alone. By analyzing feature importance, practitioners can verify that their model has learned economically sensible patterns rather than spurious correlations, instill confidence in stakeholders by explaining which real-world factors drive the predictions, and identify which data collection efforts matter most for model improvement. Lower-importance features like coal availability (305) suggest that despite being technically available in the data, they provide little predictive value in the current market dynamics, highlighting how feature importance serves as both a validation tool and a guide for future feature engineering efforts.

Create the price validation dataset 

In [31]:
valid_idx = (X_valid_solar.index
             .intersection(X_valid_wind.index)
             .intersection(X_valid_load.index))

print(f"\nValidation period indices: {len(valid_idx)} samples")


Validation period indices: 8546 samples


In [39]:
solar_pred_test = pd.Series(solar_model.predict(X_valid_solar.loc[valid_idx]), index=valid_idx, name='pred_solar')
wind_pred_test  = pd.Series(wind_model.predict(X_valid_wind.loc[valid_idx]),   index=valid_idx, name='pred_wind')
load_pred_test  = pd.Series(load_model.predict(X_valid_load.loc[valid_idx]),   index=valid_idx, name='pred_load')


In [49]:
load_pred_test

2024-01-01 20:00:00+00:00    61301.535156
2024-01-01 21:00:00+00:00    61123.644531
2024-01-01 22:00:00+00:00    62201.066406
2024-01-01 23:00:00+00:00    61500.343750
2024-01-02 00:00:00+00:00    60003.976562
                                 ...     
2024-12-31 19:00:00+00:00    74416.117188
2024-12-31 20:00:00+00:00    70104.484375
2024-12-31 21:00:00+00:00    70779.562500
2024-12-31 22:00:00+00:00    70811.570312
2024-12-31 23:00:00+00:00    69607.085938
Name: pred_load, Length: 8546, dtype: float32

In [54]:
weather_features = engineer_weather_features(prepare_weather(weather_test))

price_df_test = weather_features.join(network_train)
# price_df_test=price_df_test[[ 
  
#     'temp_mean', 'temp_std', 
#     'temp_roll_3h_mean', 'temp_roll_24h_mean',
#     'tcc_mean', 'tcc_std', 
#     'tcc_roll_3h_mean', 'tcc_roll_24h_mean', 'tcc_roll_6h_std',
#     'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
#     'EEX_CARBON', 'EEX_COAL', 'EEX_GAS_PEG',
#     'FR_availability_coal', 'FR_availability_gas', 
#     'FR_availability_hydro', 'FR_availability_nuclear',
    
# ]]

In [55]:
price_df_test

,ds,ssrd_mean,ssrd_std,tcc_mean,tcc_std,temp_mean,temp_std,wind_mean,wind_std,ssrd_lag1,...,solar_potential,EEX_CARBON,EEX_COAL,EEX_GAS_PEG,FR_capacity_solar,FR_capacity_wind,FR_availability_coal,FR_availability_gas,FR_availability_hydro,FR_availability_nuclear
24,2025-01-02 00:00:00+00:00,0.0,0.0,0.901928,0.236845,6.123503,2.876559,9.950920,4.013052,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2025-01-02 01:00:00+00:00,0.0,0.0,0.905479,0.229249,6.088181,2.853961,9.728351,4.218218,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2025-01-02 02:00:00+00:00,0.0,0.0,0.890294,0.241391,5.914766,2.921683,9.379213,4.408256,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025-01-02 03:00:00+00:00,0.0,0.0,0.902722,0.231170,5.666475,2.897085,9.146226,4.524262,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2025-01-02 04:00:00+00:00,0.0,0.0,0.927873,0.194912,5.427680,2.809566,8.723382,4.521831,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00+00:00,0.0,0.0,0.176485,0.327199,0.128238,3.175300,4.704884,2.509074,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8756,2025-12-31 20:00:00+00:00,0.0,0.0,0.183696,0.330535,-0.226904,3.120010,4.515302,2.434518,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8757,2025-12-31 21:00:00+00:00,0.0,0.0,0.189414,0.333751,-0.485759,3.055300,4.347421,2.369665,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8758,2025-12-31 22:00:00+00:00,0.0,0.0,0.200738,0.333118,-0.750103,3.053980,4.230757,2.338329,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


['temp_mean', 'temp_std', 'temp_roll_3h_mean', 'temp_roll_24h_mean', 'tcc_mean', 'tcc_std', 'tcc_roll_3h_mean', 'tcc_roll_24h_mean', 'tcc_roll_6h_std', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'EEX_CARBON', 'EEX_COAL', 'EEX_GAS_PEG', 'FR_availability_coal', 'FR_availability_gas', 'FR_availability_hydro', 'FR_availability_nuclear', 'pred_solar', 'pred_wind', 'pred_load'